In [1]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# 딕셔너리 형태의 타입(키/값 타입을 명시한 dict)을
# 정의할 수 있게 해주는 도구입니다.
from typing_extensions import TypedDict

In [2]:
# 이 예제에서는 상태를 "여러 층"으로 나누어서 사용합니다.
# - InputState: 그래프의 "입력"으로 받는 데이터 모양
# - PrivateState: 그래프 안에서만 쓰는 내부 상태
# - OutputState: 그래프가 "결과"로 내보내는 데이터 모양
# - MegaPrivate: 아주 내부적으로만 쓰는 추가 상태(예시)

# 그래프 내부에서 사용하는 상태 (a, b 라는 숫자 값 두 개)
class PrivateState(TypedDict):
  a: int
  b: int

# 그래프에 처음 들어오는 입력 상태 (hello 문자열만 있음)
class InputState(TypedDict):
  hello: str

# 그래프가 최종적으로 바깥으로 내보내는 출력 상태 (bye 문자열만 있음)
class OutputState(TypedDict):
  bye: str

# 내부에서만 쓰는 비밀 상태 (secret: bool)
class MegaPrivate(TypedDict):
  secret: bool

# StateGraph 를 만들 때,
# - 첫 번째 인자: 내부 상태의 스키마(여기서는 PrivateState)
# - input_schema: 입력으로 받는 상태의 스키마
# - output_schema: 결과로 내보내는 상태의 스키마 를 따로 지정할 수 있습니다.
graph_builder = StateGraph(
  PrivateState,
  input_schema=InputState,
  output_schema=OutputState,
)
  

In [3]:
# 여러 노드가 각각 다른 스키마 타입을 사용하도록 예시를 보여줍니다.

# node_one 은 입력 상태(InputState)를 받아서, 다시 InputState 를 반환합니다.
# 그래프 시작 부분에서 "입력"을 다루는 역할이라고 볼 수 있습니다.
def node_one(state: InputState) -> InputState:
  print("node_one ->", state)
  return {"hello": "world"}

# node_two, node_three, node_four 는 내부 상태(PrivateState)를 다룹니다.
# 그래프 안에서만 쓰이는 a, b, bye 등의 값을 채워 넣는 역할입니다.
def node_two(state: PrivateState) -> PrivateState:
  print("node_two ->", state)
  return {
    "a": 1,
  }

def node_three(state: PrivateState) -> PrivateState:
  print("node_three ->", state)
  return {
    "b": 1
  }

def node_four(state: PrivateState) -> PrivateState:
  print("node_four ->", state)
  return {
    "bye": "world"
  }

# node_five 는 출력 상태(OutputState)를 받아서,
# MegaPrivate 에 해당하는 데이터를 만들어 내는 예시입니다.
def node_five(state: OutputState):
  return {
    "secret": True
  }

# node_six 는 MegaPrivate 타입의 상태를 받아서 단순히 출력만 합니다.
def node_six(state: MegaPrivate):
  print(state)

In [4]:
# 그래프에 사용할 노드들을 등록합니다.
# 첫 번째 인자: 그래프 안에서 부를 노드 이름(문자열)
# 두 번째 인자: 실제로 실행될 파이썬 함수
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)
graph_builder.add_node("node_five", node_five)
graph_builder.add_node("node_six", node_six)

# 노드들 사이의 실행 순서를 간선(edge)으로 정의합니다.
# START -> node_one -> node_two -> node_three -> node_four
#       -> node_five -> node_six -> END
# 이런 순서로 노드들이 차례대로 호출됩니다.
graph_builder.add_edge(START, "node_one" )
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", "node_four")
graph_builder.add_edge("node_four", "node_five")
graph_builder.add_edge("node_five", "node_six")
graph_builder.add_edge("node_six", END)


In [5]:
# 지금까지 정의한 노드와 상태 스키마, 간선 정보를 이용해서
# 실제로 실행 가능한 그래프 객체를 만듭니다.
graph = graph_builder.compile()

# graph.invoke(...) 를 호출하면 그래프를 "한 번 실행" 하는 것과 같습니다.
# - 초기 입력으로 {"hello": "world"} 를 넣어 주면,
# - START 에서 시작해서 node_one ~ node_six 까지 차례대로 실행되고
# - 각 단계에서 입력/내부/출력/비밀 상태가 어떻게 바뀌는지
#   print 출력으로 확인할 수 있습니다.
graph.invoke(
  {"hello": "world"}
)

node_one -> {'hello': 'world'}
node_two -> {}
node_three -> {'a': 1}
node_four -> {'a': 1, 'b': 1}
{'secret': True}


{'bye': 'world'}